# # # LangChain 06 · 管道 → 测试护栏 → MCP 进阶 → RAG 知识库
# #
# # 这是 `02_langchain/` 里**最偏工程化**的一课：前面几课把 Agent 跑起来了，
# # 这一课回答三个「跑起来之后呢」的问题 ——
# # **怎么证明它是对的**（测试与护栏）、**怎么把它接上外部工具世界**（MCP 进阶）、
# # **怎么让它查自己的资料**（RAG）。
# #
# # 开头先用课案原版的 LCEL 管道热一下手（顺便说清它现在处于什么位置）。
# #
# # | 节 | 主题 | 关键概念 | 来源文件 |
# # |---|---|---|---|
# # | 1 | LCEL 管道（课案原版） | `prompt \| model \| parser`、`stream` / `batch` / 子链复用 | `15_管道.py`（65 行） |
# # | 2 | 测试与护栏 | 假模型单测、轨迹断言、确定性护栏、Runtime Context | `16_测试与护栏_官方补充.py`（388 行） |
# # | 3 | MCP 进阶 | 连接生命周期、多服务端命名空间、三原语、接 DeepAgents | `21_MCP进阶_官方补充.py`（440 行） |
# # | 4 | RAG 知识库 | 加载→切分→向量化→召回→交叉编码精排→作答、agentic RAG | `24_RAG知识库_官方补充.py`（390 行） |
# #
# # > **本 notebook 由 `Agent/02_langchain/` 下 4 个脚本合并而成**：
# # > `15_管道.py`、`16_测试与护栏_官方补充.py`、`21_MCP进阶_官方补充.py`、`24_RAG知识库_官方补充.py`。
# # > 四份原本各自独立、各自写运行前置；这里收敛成**一条链路**：先讲写法，再讲怎么测，再讲怎么接外部工具，最后讲怎么接自己的知识。
# #
# # **官方文档**
# # - 测试：<https://docs.langchain.com/oss/python/langchain/test/index>
# # - 单元测试：<https://docs.langchain.com/oss/python/langchain/test/unit-testing>
# # - 评估（轨迹匹配）：<https://docs.langchain.com/oss/python/langchain/test/evals>
# # - 护栏：<https://docs.langchain.com/oss/python/langchain/guardrails>
# # - Runtime / 依赖注入：<https://docs.langchain.com/oss/python/langchain/runtime>
# # - MCP 总览：<https://docs.langchain.com/oss/python/langchain/mcp/index>
# # - MCP 连接生命周期：<https://docs.langchain.com/oss/python/langchain/mcp/connections>
# # - 知识库（RAG）：<https://docs.langchain.com/oss/python/langchain/knowledge-base>
#

# # ## 运行条件
# #
# # | 项 | 说明 |
# # |---|---|
# # | 🔴 运行档位 | **需外部服务** —— 本 notebook 会**自己拉起**一个 HTTP MCP 服务端（子进程，端口 8110），并真实调用模型网关与 SiliconFlow 的 embedding / rerank |
# # | 依赖 | `langchain` / `langgraph` / `langchain-mcp-adapters` / `fastmcp` / `deepagents` / `uvicorn`（本项目 venv 已装） |
# # | 密钥 | `settings.api_key`（模型网关）、`settings.embedding.*` / `settings.rerank.*`（SiliconFlow）—— 全部来自仓库根的 `.env` |
# # | 前置服务 | **无外部常驻服务**：MCP 服务端在第 3 节现场拉起、在**该节最后一格**用 `taskkill /F /T` 连子进程树一起关掉，不会留端口 |
# # | 预计耗时 | 约 2~4 分钟（第 4 节的作答与 agentic RAG 占大头；第 2 节**完全离线**、0 次模型调用） |
# # | 可选包 | `agentevals`（**未装也能跑** —— 第 2 节会打印中文提示并退回本地简化版匹配器） |
# # | 环境 | 本机开着 Clash 等系统代理，回环请求会被接管 —— notebook 内已设 `NO_PROXY=127.0.0.1,localhost` |
# #
# # 四节对外部条件的依赖并不一样，心里有个数就不会白等：
# #
# # | 节 | 要模型 | 要服务 | 要 SiliconFlow |
# # |---|---|---|---|
# # | 1. LCEL 管道 | ✅ | ❌ | ❌ |
# # | 2. 测试与护栏 | ❌（全用假模型） | ❌ | ❌ |
# # | 3. MCP 进阶 | ✅ | ✅ HTTP MCP（自动起停） | ❌ |
# # | 4. RAG 知识库 | ✅ 作答模型 | ❌ | ✅ 向量化 + 精排 |
# #
# # > ⚠️ **关于下面所有「预期输出」块**：它们是本机**某一次真实运行的逐字记录**。
# # > 凡是有模型作答的地方（第 1 节的回答、第 3 节 Demo 1/4、第 4 节 Demo 4/5），
# # > **文字措辞每次运行都不同**；相似度/rerank 分数、耗时、token 数也会小幅浮动。
# # > 稳定的是**结构与量级**：维度（1024）、工具是否被发现、模型是否真的调了工具、
# # > 「模型调用次数」这类计数断言 —— 所以**别拿文字去对答案，对结构**。
# # > 第 2 节是唯一**完全确定性**的一节（假模型 + 断言），它的输出每次一模一样。
#

# # ## 本节地图
# #
# # 从「写法」到「验收」到「接外部」到「接自己的知识」，一条线走到底：
# #
# # ```mermaid
# # graph LR
# #     A["1. LCEL 管道<br/>prompt → model → parser"] --> B["2. 测试与护栏<br/>假模型 / 轨迹 / 护栏 / Context"]
# #     B --> C["3. MCP 进阶<br/>stdio ×2 + HTTP 子进程"]
# #     C --> D["4. RAG 知识库<br/>召回 + 精排 + agentic RAG"]
# #     C --> E["3 节末：taskkill<br/>关掉 HTTP MCP 服务端"]
# # ```
# #
# # 等价表格（裸 JupyterLab 不渲染 mermaid，看表即可）：
# #
# # | 步 | 在干什么 | 用到的东西 | 产出 |
# # |---|---|---|---|
# # | 1 | 用 `\|` 把三段串成管道 | `ChatPromptTemplate` / `init_chat_model` / `StrOutputParser` | 一个可 `invoke` / `stream` / `batch` 的链 |
# # | 2 | 证明 Agent 是对的 | `GenericFakeChatModel` / `ScriptedModel` / `wrap_model_call` / `ToolRuntime` | 4 个**离线**可跑的断言 Demo |
# # | 3 | 把 MCP 工具接进来 | `MultiServerMCPClient` / `FastMCP` 服务端 / `create_deep_agent` | agent 真的调到了 MCP 工具 |
# # | 4 | 让模型查自己的资料 | `OpenAIEmbeddings`（bge-m3）/ `InMemoryVectorStore` / 自包 reranker | 召回→精排→作答→工具化检索 |
# #
# # **与上下节的衔接**：
# # - 上游：`01_langgraph/` 给了「编排」的正统答案（StateGraph），`02_langchain/` 前面几课给了 Agent 的全部零件（模型/消息/工具/记忆/中间件/人工审核）。
# # - 下游：RAG 这一节的 `store` 与 `reranker` 正是 `RAG/` 项目（Milvus + 双路召回）里那套东西的**官方 LangChain 写法**最小版。
#

# # ## 0. 环境引导
# #
# # notebook 的工作目录默认是它自己所在的文件夹，而 `config.py` 在仓库根 ——
# # 少了下面这一格，后面每次 `from config import settings` 都会 `ModuleNotFoundError`。
# #
# # 它顺便给出两个后面要用的变量：
# # - `NB_DIR`：notebook 所在目录（第 6 节里 `__file__` 的替代品）；
# # - `WORKDIR`：本 notebook 的临时工作目录（已被 `.gitignore` 覆盖）。
# #
# # > ⚠️ `WORKDIR`（`tmp_nb_work/`）是**同一章共享**的容器。本课再往下套一层
# # > `mcp_advanced/` 专属子目录，免得和同章其它 notebook 并发执行时互相踩文件。
#

In [ ]:
# ===== 环境引导（每个 notebook 的第一格，不要改）=====
import os
import sys
from pathlib import Path

NB_DIR = Path.cwd()                 # notebook 所在目录（chdir 之前先抓住）
ROOT = NB_DIR
while not (ROOT / "config.py").exists():
    if ROOT.parent == ROOT:
        raise RuntimeError("没找到 config.py：请在 Python_Base 仓库内运行本 notebook")
    ROOT = ROOT.parent

os.chdir(ROOT)                      # 让相对路径（data/、output.txt 等）都相对仓库根
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

WORKDIR = NB_DIR / "tmp_nb_work"    # 本 notebook 的临时工作目录（已被 .gitignore 覆盖）
WORKDIR.mkdir(exist_ok=True)

print("仓库根：", ROOT)
print("临时目录：", WORKDIR)

# # ### 0.1 前置条件自检
# #
# # 把「缺什么」提前说清楚，比跑到一半报 `AttributeError` 友好得多。
# # 这一格**不发任何外部请求**，只查本地：包能不能导入、`.env` 里的字段有没有值、8110 端口空不空。
# #
# # 缺依赖或缺密钥时会打印 `[跳过]` 开头的**中文提示**（执行器会把它识别为「降级」而不是失败），
# # 但本机这几项都是齐的，正常应当一路 ✔。
#

In [ ]:
# ===== 前置条件自检（不联网、不调模型）=====
import importlib.util
import socket

from config import settings

_missing = []
for _mod in ("langchain", "langgraph", "langchain_mcp_adapters", "fastmcp", "deepagents", "uvicorn"):
    # ⚠️ 这里用 find_spec 而不是真的 import：第 3 节必须**在导入 langchain_mcp_adapters 之前**
    # 换掉 sys.stderr（原因见 3.1 那一格的说明），所以这一格绝不能把它提前导进来。
    if importlib.util.find_spec(_mod) is None:
        _missing.append(_mod)

if _missing:
    print("[跳过] 以下依赖缺失，本 notebook 的部分小节跑不了：")
    for _m in _missing:
        print("   -", _m)
    print("   本课不自动装包；请自己在 venv 里补齐后重跑")
else:
    print("✔ 第 3 节所需依赖齐全（langchain_mcp_adapters / fastmcp / deepagents / uvicorn）")

if settings.api_key:
    print("✔ 作答模型：", settings.model_name, "@", settings.base_url)
else:
    print("[跳过] .env 里没有 settings.api_key —— 第 1 / 3 / 4 节无法调用模型")

print("✔ 向量化模型：", settings.embedding.model, "@", settings.embedding.base_url)
print("✔ 精排模型  ：", settings.rerank.model, "@", settings.rerank.base_url)

try:
    import agentevals  # noqa: F401
    print("✔ agentevals 已安装 —— 第 2 节 Demo 2 可用官方四模式（strict/unordered/subset/superset）")
except ImportError:
    print("· 可选包 agentevals 未安装 —— 第 2 节 Demo 2 会退回本地简化版匹配器（不影响结论）")

with socket.socket() as _sock:
    _sock.settimeout(0.3)
    _port_free = _sock.connect_ex(("127.0.0.1", 8110)) != 0
print(f"✔ 第 3 节要用的 MCP HTTP 端口 8110：{'空闲' if _port_free else '已被占用 —— 起服务端会失败，请先释放'}")

# # ## 1. 课案原版：LCEL 管道（65 行）
# #
# # 原版 `15_管道.py` 只有 65 行，正好把 LCEL（LangChain Expression Language）的
# # 三种接口各出现一次：
# #
# # | 接口 | 语义 | 本节的用法 |
# # |---|---|---|
# # | `invoke` | 一次输入 → 一次输出 | 问一个问题，拿一个答案 |
# # | `stream` | 一次输入 → 逐 token 输出 | 打字机效果 |
# # | `batch` | 一批输入 → 一批输出 | 并行跑多个问题 |
# #
# # ⚠️ **历史遗留提醒（很重要，别把它当主线学）**
# #
# # 官方**新文档全站搜索 "LCEL" 是零命中** —— LCEL 已经**不是官方主线**了。
# # 官方现在「编排」的对应物是 **LangGraph 的 `StateGraph`**：把 `create_agent` 的产物当节点，
# # 和确定性步骤混编（见 `multi-agent/custom-workflow` 与 `01_langgraph/` 那一章）。
# #
# # 那为什么这一节还在？两个现实理由：
# #
# # 1. 存量代码里 `|` 的写法**到处都是**，读不懂它就维护不了老项目；
# # 2. `prompt | model | parser` 这 30 秒就能说清的心理模型，是理解
# #    「组件 = 可调用对象，上一个的输出喂给下一个」的最短路径 —— 后面 Agent /
# #    Middleware / 工具链的心智模型都是从这里长出来的。
# #
# # **新写编排逻辑时优先用 LangGraph；`|` 用来拼「一条直线的数据流」仍然方便。**
#

# # ### 1.1 模型与三段管道的零件
# #
# # `init_chat_model` 是「一个入口接多家厂商」的统一构造器：`model_provider` 选协议，
# # `base_url` 指向兼容端点。参数全部来自 `.env`（经 `config.py` 收敛），
# # 所以换模型只改 `.env`，不动代码。
#

In [ ]:
from langchain.chat_models import init_chat_model
from langchain_core.output_parsers import StrOutputParser
from langchain_core.prompts import ChatPromptTemplate

llm = init_chat_model(
    model_provider="openai",
    model=settings.model_name,
    api_key=settings.api_key,
    base_url=settings.base_url,
)

# # ### 1.2 三段管道：模板 → 模型 → 解析器
# #
# # `chain = prompt | llm | parser` 数据从左流向右：
# #
# # | 段 | 输入 | 输出 |
# # |---|---|---|
# # | `ChatPromptTemplate` | `{"domain": ..., "question": ...}` | 一组 `ChatPromptValue` |
# # | `llm` | 消息列表 | `AIMessage` |
# # | `StrOutputParser` | `AIMessage` | 纯字符串 |
# #
# # 最后一段是必须的吗？不是 —— 但少了它，拿到手的是 `AIMessage` 对象，
# # 打印出来带着 `content=` / `additional_kwargs=` 一堆元数据；解析器把它剥成用户真正要看的那句话。
#

In [ ]:
prompt = ChatPromptTemplate.from_messages(
    [
        ("system", "你是{domain}专家，回答不超过 30 字。"),
        ("user", "{question}"),
    ]
)
parser = StrOutputParser()  # 把 AIMessage 解析成纯字符串

chain = prompt | llm | parser

result = chain.invoke({"domain": "数据库", "question": "什么是索引？"})
print("管道结果：", result)

# # ### 1.3 同一个链，换成流式
# #
# # `stream` 返回**生成器**，每吐一个片段就打印一次 —— 这就是所有「打字机效果」的实现原理。
# # 注意链本身**没有任何改动**：接口是 `Runnable` 协议的一部分，换方法不换对象。
#

In [ ]:
print("----- 流式 -----")
for token in chain.stream({"domain": "后端", "question": "什么是消息队列？"}):
    print(token, end="", flush=True)
print()

# # ### 1.4 同一个链，换成批量并行
# #
# # `batch` 接收**输入列表**，返回输出列表。它内部并发调用（不是 for 循环串行），
# # 所以 10 个问题的总耗时接近 1 个问题 —— 这是 LCEL 白送的能力。
#

In [ ]:
results = chain.batch(
    [
        {"domain": "前端", "question": "什么是虚拟 DOM？"},
        {"domain": "运维", "question": "什么是容器？"},
    ]
)
for r in results:
    print("批量结果：", r)

# # ### 1.5 管道串联管道：子链复用
# #
# # 链本身也是 `Runnable`，所以能**当零件再用**：`analyzer | (lambda text: ...)`。
# #
# # 这里塞进一个普通函数是最能说明问题的一手 —— 说明「管道」并不神秘，
# # 它只是「上一个的输出 = 下一个的输入」这条约定：
# # 框架组件和你的普通函数**在管道里地位相同**。
#

In [ ]:
analyzer = prompt | llm | parser
summary_chain = analyzer | (lambda text: f"【分析摘要】{text}")
print(summary_chain.invoke({"domain": "AI", "question": "什么是 RAG？"}))

# # ## 2. 测试与护栏（来自 `16_测试与护栏_官方补充.py`）
# #
# # 第 1 节回答「怎么写」，这一节回答**「怎么证明写对了」**—— 这是官方文档里
# # 课案完全空白的「工程化交付」维度。对应四处官方出处：
# #
# # | 官方文档 | 讲什么 | 本节的 Demo |
# # |---|---|---|
# # | `test/unit-testing.mdx` | 单测不该依赖外部服务 | Demo 1 |
# # | `test/evals.mdx` | 轨迹匹配（trajectory match） | Demo 2 |
# # | `guardrails.mdx` | 确定性护栏 vs 模型护栏 | Demo 3 |
# # | `runtime.mdx` | Runtime Context 依赖注入 | Demo 4 |
# #
# # ### 为什么第 2 节**全离线**（0 次真实模型调用）
# #
# # 因为测试与护栏的价值恰恰在于**可重复、可断言** —— 依赖真模型反而测不稳。
# # 官方单测文档自己推荐的 `GenericFakeChatModel` 就是零 API 的方案。
# #
# # ### 官方缺口表（这一节覆盖第 1、3、8 项）
# #
# # | # | 缺口 | 官方出处 | 优先级 |
# # |---|---|---|---|
# # | 1 | 测试与评估（单测/集成/轨迹评估） | `test/*.mdx` | 高 ← **本节的 Demo 1/2** |
# # | 2 | RAG / 语义检索 | `knowledge-base.mdx` | 高 ← 第 4 节 |
# # | 3 | Runtime Context 与依赖注入 | `runtime.mdx` | 高 ← **本节的 Demo 4** |
# # | 4 | Skills 渐进披露（多 Agent 第 5 种模式） | `multi-agent/skills.mdx` | 高（要模型） |
# # | 5 | LangGraph 自定义工作流 | `multi-agent/custom-workflow.mdx` | 高（要模型） |
# # | 6 | 事件流 v3（`stream_events` 类型化投影） | `event-streaming.mdx` | 中（要模型） |
# # | 7 | 上下文工程总纲 | `context-engineering.mdx` | 中（要模型） |
# # | 8 | Guardrails 安全护栏 | `guardrails.mdx` | 中 ← **本节的 Demo 3** |
# # | 9 | MCP 进阶（连接生命周期/认证/Elicitation） | `mcp/connections.mdx` | 中 ← 第 3 节 |
# # | 10 | 模型配置进阶（多模态/限流/token 用量） | `models.mdx` | 中（部分要视觉模型） |
# # | 11 | 可观测与可视化调试（LangSmith/Studio） | `observability.mdx` | 中低（要外部账号） |
# # | 12 | Deep Agents harness 组装教程 | `deep-agent-from-scratch.mdx` | 中低 |
# #
# # 完整对照表见 `Agent/官方文档缺口对照.md`。
#

# # ### 2.1 工具箱：两个假模型
# #
# # 断网也能跑的前提，是**把模型换掉**。这里准备两把锤子，用途不同：
# #
# # | 工具 | 是什么 | 什么时候用 |
# # |---|---|---|
# # | `GenericFakeChatModel` | 现成的「消息列表播放器」：按顺序吐预设的 `AIMessage` | 官方单测推荐；只关心框架行为时（Demo 1） |
# # | `ScriptedModel` | 继承 `ChatOpenAI`、**只覆写 `_generate`** 的剧本模型 | 需要模型「调工具」、或要盯住模型**实际收到了什么**时（Demo 2/3/4） |
# #
# # `ScriptedModel` 的关键点：它继承真的 `ChatOpenAI`，所以 `bind_tools`、消息校验等
# # 框架方法**全部沿用真实现**，唯一被替换的是「真正发 HTTP 请求」那一步。
# # 于是 `api_key="offline"` / `base_url="http://localhost:9"` 这种假值永远不会被用到。
# #
# # `_received` 记录每次模型**实际收到**的消息列表 —— Demo 4 就靠它验证「上下文有没有真的注入进去」。
#

In [ ]:
from dataclasses import dataclass
from typing import Any

from langchain.agents import create_agent
from langchain.agents.middleware import wrap_model_call
from langchain.tools import ToolRuntime, tool
from langchain_core.language_models import GenericFakeChatModel
from langchain_core.messages import AIMessage
from langchain_core.outputs import ChatGeneration, ChatResult
from langchain_core.tools import tool as core_tool
from langchain_openai import ChatOpenAI
from langgraph.checkpoint.memory import InMemorySaver
from pydantic import PrivateAttr

In [ ]:
def ai_tool_call(name: str, args: dict, call_id: str) -> AIMessage:
    """构造一条「模型要调工具」的 AIMessage（脚本模型的一行剧本）。"""
    return AIMessage(
        content="",
        tool_calls=[{"name": name, "args": args, "id": call_id, "type": "tool_call"}],
    )


class ScriptedModel(ChatOpenAI):
    """按剧本依次吐消息的假模型（与 11_内置中间件_官方补充.py 里的同名类同一手法）。

    继承 ChatOpenAI、只覆写 _generate —— bind_tools / 消息校验等框架方法沿用真实现，
    唯一被替换的是「真正发 HTTP 请求」那一步，所以断网也能跑。
    _received 记录每次模型**实际收到**的消息列表（Demo 4 靠它验证注入是否生效）。
    """

    _script: list = PrivateAttr(default_factory=list)
    _cursor: int = PrivateAttr(default=0)
    _received: list = PrivateAttr(default_factory=list)

    def _generate(self, messages, stop=None, run_manager=None, **kwargs):
        self._received.append(list(messages))
        message = self._script[self._cursor]
        self._cursor += 1
        return ChatResult(generations=[ChatGeneration(message=message)])


def make_scripted(script: list) -> ScriptedModel:
    """造一个剧本模型。api_key / base_url 传假值即可：永远不会被真正用到。"""
    model = ScriptedModel(model="scripted", api_key="offline", base_url="http://localhost:9")
    model._script = script
    return model


def extract_trajectory(result: dict) -> list[str]:
    """从结果消息里抽出「工具调用轨迹」：模型依次调了哪些工具（官方轨迹评估的核心数据）。"""
    trajectory: list[str] = []
    for message in result["messages"]:
        for call in getattr(message, "tool_calls", None) or []:
            trajectory.append(call["name"])
    return trajectory


def run_demo(_demo) -> None:
    """逐 Demo 兜底：网关抖动时打印中文提示并继续，而不是让整跑崩掉。

    源文件把这段 try/except 写在 `if __name__ == "__main__"` 的循环里；
    notebook 里换成函数，好让每个 Demo 各占一格、各自有「预期输出」。
    """
    try:
        _demo()
    except Exception as _exc:  # noqa: BLE001
        print(f"\n  ⚠️ {_demo.__name__} 本次未跑完（网关抖动/超时，非代码问题）："
              f"{type(_exc).__name__}")
        print("  重跑一次通常即可。")

# # ### 2.2 Demo 1：官方单测模式 —— 假模型 + 真 checkpointer
# #
# # 官方 `unit-testing.mdx` 的核心思路一句话：**单元测试不该依赖外部服务**。
# #
# # 这一格的三个零件：
# #
# # - `GenericFakeChatModel(messages=iter([...]))`：预设两轮回复的「播放器」；
# # - `InMemorySaver()`：当 checkpointer 用，于是**多轮对话、记忆有没有保住都能断言**；
# # - `assert`：断言的是**框架行为**（记忆有没有带上、顺序对不对），不是模型说了什么漂亮话。
# #
# # > ⚠️ 坑：`GenericFakeChatModel` 的 `messages` 是**一次性迭代器** —— 每 `invoke` 一次消费一条，
# # > 剧本用完再 invoke 会直接 `StopIteration`。这是官方示例里最容易踩的小坑，所以每个用例都要**重建模型实例**。
#

In [ ]:
def demo_1_fake_model_unit_test() -> None:
    print("=" * 70)
    print("Demo 1：官方单测模式 —— GenericFakeChatModel + InMemorySaver")
    print("=" * 70)

    fake = GenericFakeChatModel(
        messages=iter([
            AIMessage(content="好的，我记住了：你叫小明。"),
            AIMessage(content="你叫小明。"),
        ])
    )
    agent = create_agent(model=fake, tools=[], checkpointer=InMemorySaver())
    config = {"configurable": {"thread_id": "unit-test-1"}}

    first = agent.invoke({"messages": [{"role": "user", "content": "我叫小明"}]}, config)
    assert first["messages"][-1].content == "好的，我记住了：你叫小明。", first["messages"][-1]
    print(f"  第 1 轮回复：{first['messages'][-1].content}")

    second = agent.invoke({"messages": [{"role": "user", "content": "我叫什么？"}]}, config)
    assert second["messages"][-1].content == "你叫小明。"
    contents = [str(m.content) for m in second["messages"]]
    assert "我叫小明" in contents, contents
    print(f"  第 2 轮回复：{second['messages'][-1].content}")
    print(f"  第 2 轮状态里共 {len(second['messages'])} 条消息，第 1 轮的用户消息仍在 → 记忆断言通过")
    print(
        "  ↑ 这就是官方推荐的单元测试形态：断言的是**框架行为**（记忆有没有带上、\n"
        "    顺序对不对），而不是模型说了什么漂亮话 —— 所以它永远不会 flaky。"
    )


run_demo(demo_1_fake_model_unit_test)

# # ### 2.3 Demo 2：轨迹断言 —— 断言「做了什么」，不是「说了什么」
# #
# # 官方 `test/evals.mdx` 用 `agentevals` 包做 **trajectory match**，四种模式：
# #
# # | 模式 | 语义 | 记忆口诀 |
# # |---|---|---|
# # | `strict` | 结构与顺序**完全一致** | 一模一样 |
# # | `unordered` | 内容一致，**顺序无关** | 集合相等 |
# # | `subset` | 实际**只允许调**参考里的工具，**不许多调** | 「子集」= 实际 ⊆ 参考 |
# # | `superset` | 实际**至少包含**参考里的工具，**允许多调** | 「超集」= 实际 ⊇ 参考 |
# #
# # > ⚠️ **`subset` / `superset` 最容易记反**，因为「谁是谁的子集」在两种写法里都说得通。
# # > 请按上表的**方向**记：**参考是基准**，`subset` 要求实际不比参考多，`superset` 允许实际比参考多。
# #
# # 本仓库**没装 `agentevals`** —— 按仓库惯例：缺包给出中文提示，同时用**本地简化版**
# # 把同样的思想演示出来。真正的评估逻辑并不神秘，就是**比对工具调用序列**：
# # `strict_match` 是列表全等，`subset_match` 是「每个期望项都能在实际序列里找到且只用一次」。
# #
# # 还有一件必须做的事：**反面用例**。期望与实际不符时匹配器必须**能识别出来** ——
# # 否则测试是「假绿」，比没有测试更危险。
#

In [ ]:
def demo_2_trajectory_assertion() -> None:
    print("\n" + "=" * 70)
    print("Demo 2：轨迹断言 —— Agent 该调哪个工具、按什么顺序")
    print("=" * 70)

    try:
        import agentevals  # noqa: F401

        print("  检测到 agentevals：生产评估可用官方的 trajectory match 四模式")
    except ImportError:
        print("  （未安装 agentevals —— 装了才有官方四模式：strict/unordered/subset/superset）")
        print("  本 Demo 用本地简化版演示同样的思想；要装：uv add agentevals")

    @core_tool
    def get_weather(city: str) -> str:
        """查询指定城市的天气。"""
        return f"{city}：晴，25℃"

    # 剧本：模型发起一次工具调用，然后给出最终答复
    model = make_scripted([
        ai_tool_call("get_weather", {"city": "北京"}, "c1"),
        AIMessage(content="北京今天晴，25℃。"),
    ])
    agent = create_agent(model=model, tools=[get_weather])
    result = agent.invoke({"messages": [{"role": "user", "content": "北京天气如何？"}]})

    actual = extract_trajectory(result)
    print(f"  实际轨迹（模型依次调用的工具）：{actual}")

    # ---- 本地简化版匹配器：strict（全等）与 subset（子集）----
    def strict_match(expected: list[str], got: list[str]) -> bool:
        """顺序与内容完全一致。"""
        return expected == got

    def subset_match(expected: list[str], got: list[str]) -> bool:
        """期望的每一步都在实际序列里出现过（允许实际多调了别的工具）。"""
        remaining = list(got)
        for name in expected:
            if name not in remaining:
                return False
            remaining.remove(name)   # 每个期望项只能匹配一次
        return True

    expected = ["get_weather"]
    assert strict_match(expected, actual), (expected, actual)
    assert subset_match(expected, actual), (expected, actual)
    print(f"  strict 匹配 {expected} → 通过（顺序与内容全等）")
    print(f"  subset 匹配 {expected} → 通过（期望步骤都在实际里）")

    # ---- 反面用例：期望与实际不符时必须**能识别出来**（否则测试是假绿）----
    wrong_expected = ["search_documents"]
    assert not strict_match(wrong_expected, actual)
    assert not subset_match(wrong_expected, actual)
    print(f"  反面用例 strict/subset 匹配 {wrong_expected} → 正确判为不一致 ✔")
    print(
        "  ↑ 轨迹断言的意义：模型「换个方式问同样的问题」时，回复文本每次都不同，\n"
        "    但它**该走的流程**是稳定的 —— 把流程写成断言，才是可维护的 Agent 测试。"
    )


run_demo(demo_2_trajectory_assertion)

# # ### 2.4 Demo 3：确定性护栏 —— 在花钱之前拦住
# #
# # 官方 `guardrails.mdx` 把护栏分成两条路线：
# #
# # | 路线 | 手段 | 特点 | 适合 |
# # |---|---|---|---|
# # | **确定性护栏** | 正则、关键词、白名单 | 快、免费、可解释 | 已知的坏输入（本 Demo） |
# # | **模型护栏** | 让另一个模型判安全 | 灵活，但慢且要花钱 | 开放式风险 |
# #
# # 实现手段就是课案第 10 章学过的**包裹式钩子**：在 `wrap_model_call` 里检查输入，
# # 命中就**短路** —— 不调 `handler`，直接返回预设回复。
# #
# # 这一格的证据是**模型调用次数**：
# #
# # - 用例 A（正常输入）→ 放行，`model_calls` 变成 1；
# # - 用例 B（含「转账」）→ 短路，`model_calls` **保持 1**，即**这一轮模型 0 次调用**。
# #
# # `assert guard_stats["model_calls"] == 1 and guard_stats["blocked"] == 1` 把这件事钉死成测试：
# # 护栏的价值不只是「拦住了」，而是**在花钱之前就拦住**。
#

In [ ]:
FORBIDDEN_WORDS = ("转账", "信用卡号", "身份证号")
guard_stats = {"model_calls": 0, "blocked": 0}


@wrap_model_call
def safety_guard(request, handler):
    """确定性护栏：命中违禁词就短路；否则原样放行。"""
    last_message = request.messages[-1] if request.messages else None
    text = str(getattr(last_message, "content", ""))
    if any(word in text for word in FORBIDDEN_WORDS):
        guard_stats["blocked"] += 1
        print(f"    [safety_guard] 命中违禁词，拦截（本次不调用模型）")
        # 短路：直接返回一条预设回复，handler（真正调模型的那层）根本不会被调用
        return AIMessage(content="抱歉，这个请求涉及敏感信息，我不能处理。请通过人工客服渠道办理。")
    guard_stats["model_calls"] += 1
    return handler(request)

In [ ]:
def demo_3_deterministic_guardrail() -> None:
    print("\n" + "=" * 70)
    print("Demo 3：确定性护栏 —— 违禁词短路，模型 0 次调用")
    print("=" * 70)

    guard_stats["model_calls"] = 0
    guard_stats["blocked"] = 0
    model = make_scripted([
        AIMessage(content="好的，我来介绍一下转账的一般流程。"),   # 只有正常输入才会用到这条剧本
    ])
    agent = create_agent(model=model, tools=[], middleware=[safety_guard])

    # ---- 用例 A：正常输入 → 放行，模型被调 1 次 ----
    normal = agent.invoke({"messages": [{"role": "user", "content": "介绍一下 LangChain 是什么"}]})
    print(f"  用例 A（正常输入）→ 回复：{str(normal['messages'][-1].content)[:40]}")
    print(f"    模型调用次数：{guard_stats['model_calls']}（应该 1）")

    # ---- 用例 B：违禁输入 → 短路，模型调用次数**不增加** ----
    blocked = agent.invoke({"messages": [{"role": "user", "content": "帮我转账 100 万到这个账号"}]})
    print(f"  用例 B（含「转账」）→ 回复：{blocked['messages'][-1].content}")
    print(f"    模型调用次数：{guard_stats['model_calls']}（仍是 1 → 模型确实没被调用）")
    print(f"    拦截次数：{guard_stats['blocked']}")
    assert guard_stats["model_calls"] == 1 and guard_stats["blocked"] == 1
    print(
        "  ↑ 护栏的价值不只是「拦住了」，而是**在花钱之前就拦住**"
        "（用例 B 的模型调用为 0，累计那 1 次是用例 A 的）；\n"
        "    确定性护栏适合处理已知的坏模式，开放式风险再叠加模型护栏（官方两条路线并用）。"
    )


run_demo(demo_3_deterministic_guardrail)

# # ### 2.5 Demo 4：Runtime Context —— 工具怎么拿到「本次运行的上下文」
# #
# # 官方 `runtime.mdx` 的核心是**依赖注入**。Runtime 对象带着五类信息：
# #
# # | 内容 | 是什么 |
# # |---|---|
# # | `context` | 本次运行的静态上下文（用户 ID、数据库连接…） |
# # | `store` | 长期记忆 |
# # | stream writer | 自定义流 |
# # | 执行信息 | `thread_id`、`run_id` |
# # | server info | 服务端信息 |
# #
# # 工具里想读 `context`，就在签名上加 `runtime: ToolRuntime` 参数 ——
# # 它是**保留参数**，不会出现在给模型的 schema 里（模型看不到它）。
# #
# # > 同理，别把自己的参数命名成 `runtime` / `config`（官方 `tools.mdx` 的保留字表）。
# #
# # 这一格有两个用例，第二个是**踩坑实测**：
# #
# # - 用例 A：`invoke(..., context=UserContext(user_id="u-1001", tier="黄金"))` → 工具读到 `u-1001`；
# # - 用例 B：**忘了传 `context`** → `runtime.context` 是 `None`，工具里一访问属性就 `AttributeError`。
# #
# # 重点在用例 B 的「然后呢」：`ToolNode` 默认**只把 `ToolInvocationError` 转成错误消息**，
# # 其余异常（含普通 `ToolException`）一律**往外抛**，所以整个运行会**中断**，
# # 而不是变成一条模型能看见的错误消息。想让这类异常转成消息、让模型自己补救 → 挂 `ToolErrorMiddleware`
# # （写法见 `02_langchain/11_内置中间件_官方补充.py` 的 Demo 1）。
#

In [ ]:
@dataclass
class UserContext:
    """本次运行的上下文（官方叫 context_schema）：多用户/多租户的标准做法。"""

    user_id: str
    tier: str


@tool
def get_my_profile(runtime: ToolRuntime) -> str:
    """读取当前用户的档案（演示工具的依赖注入）。"""
    # runtime.context 就是 invoke(context=...) 传进来的那个对象
    return f"用户 {runtime.context.user_id}，会员等级 {runtime.context.tier}"

In [ ]:
def demo_4_runtime_context() -> None:
    print("\n" + "=" * 70)
    print("Demo 4：Runtime Context —— 工具通过 ToolRuntime 读取注入的上下文")
    print("=" * 70)

    model = make_scripted([
        ai_tool_call("get_my_profile", {}, "c1"),
        AIMessage(content="已读取到您的档案。"),
    ])
    agent = create_agent(
        model=model,
        tools=[get_my_profile],
        context_schema=UserContext,   # 声明上下文的形状
    )

    # ---- 用例 A：正常注入 ----
    result = agent.invoke(
        {"messages": [{"role": "user", "content": "我的档案是什么？"}]},
        context=UserContext(user_id="u-1001", tier="黄金"),
    )
    tool_outputs = [m for m in result["messages"] if m.type == "tool"]
    assert tool_outputs and "u-1001" in str(tool_outputs[0].content), tool_outputs
    print(f"  工具返回：{tool_outputs[0].content}")
    print("  ↑ 用户身份是 invoke 时注入的，工具代码里没有任何硬编码 —— 这就是依赖注入")

    # ---- 用例 B：忘了传 context 会怎样？（实测行为，防止生产里漏传）----
    model = make_scripted([
        ai_tool_call("get_my_profile", {}, "c1"),
        AIMessage(content="（这一轮不该正常完成）"),
    ])
    agent = create_agent(model=model, tools=[get_my_profile], context_schema=UserContext)
    try:
        agent.invoke({"messages": [{"role": "user", "content": "我的档案是什么？"}]})
        print("\n  用例 B（不传 context）→ 竟然跑完了（不符合预期）")
    except Exception as exc:  # noqa: BLE001
        print(f"\n  用例 B（不传 context）→ 抛 {type(exc).__name__}: {exc}")
        print(
            "  ↑ 实测行为（重要）：runtime.context 是 **None**，工具里一访问属性就 AttributeError；\n"
            "    而 ToolNode 默认只把 ToolInvocationError 转成错误消息，**其余异常一律往外抛**\n"
            "    （源码 langgraph/prebuilt/tool_node.py 的 _default_handle_tool_errors），\n"
            "    所以整个运行会中断，而不是变成一条模型能看见的错误消息。\n"
            "    想让这类异常转成消息、让模型自己补救 → 挂 ToolErrorMiddleware，\n"
            "    写法见 02_langchain/11_内置中间件_官方补充.py 的 Demo 1。"
        )


run_demo(demo_4_runtime_context)

# # 四个 Demo 都过完之后，`16_测试与护栏_官方补充.py` 的**实测结论**可以逐条对上：
# #
# # | 结论 | 本节的证据 |
# # |---|---|
# # | Demo 1：假模型 + `InMemorySaver` 两轮断言通过（记忆确实被带上） | 第 2 轮状态里仍留着第 1 轮的用户消息 |
# # | Demo 2：轨迹 == `["get_weather"]`，`strict`/`subset` 与反面用例都符合预期 | 「反面用例 … 正确判为不一致 ✔」 |
# # | Demo 3：违禁词短路后模型调用次数保持 1（用例 A 用掉的那次） | 用例 B 之后仍打印「模型调用次数：1」 |
# # | Demo 4：`ToolRuntime` 注入生效；**不传 context 抛 `AttributeError`** | 用例 B 打印异常类型与原因链 |
# #
# # 另外两条**只报不修**的已知现象：
# # 1. 传 `context` 时会打印两条 pydantic `UserWarning`（`PydanticSerializationUnexpectedValue`，
# #    源自 dataclass context 的序列化探测）—— **无害噪音**，不影响运行结果；
# # 2. 源文件注释里写「本文件虽有假模型为主，但 Demo 4 依赖真实模型」—— 这句**已经过期**：
# #    本节的 Demo 4 用的也是 `make_scripted` 假模型，全程 0 次真实调用。
#

In [ ]:
print("\n全部 Demo 执行完毕（0 次真实模型调用，离线可复现）。")

# # ## 3. MCP 进阶（来自 `21_MCP进阶_官方补充.py`）
# #
# # 这一节覆盖缺口表里的**两格**：LangChain 第 9 项 + DeepAgents 第 8 项。
# #
# # ### 与 `05_mcp` 章的分工
# #
# # | 那一章讲 | 这一节讲 |
# # |---|---|
# # | MCP **协议本身**：服务端怎么写、客户端怎么连、工具/资源/提示词三原语、JWT 权限、部署调试 | **连接层与工程接入**：连接生命周期、多服务端聚合与命名空间、把 MCP 工具接进 `create_agent` 与 `create_deep_agent`、返回值的真实形态 |
# #
# # ### ⚠️ 本机现实（决定了这一节为什么这么写）
# #
# # 官方**新** API 是 `langchain.mcp.MCPAdapter`（要求 `langchain[mcp]>=1.4.0`，beta）。
# # **本机实测它不可用**：
# #
# # ```text
# # ImportError: No module named 'fastmcp.client.group'
# # ```
# #
# # 原因是本机 `fastmcp` 是 3.4.7，而 `langchain.mcp` 需要 fastmcp 4.x
# # （本地 banner 已在提示 "Update available: 4.0.4"）。
# # 升级 fastmcp 会改动 `pyproject.toml` / `uv.lock`（本课**不新增依赖**），
# # 所以这一节改用**已安装的经典适配器** `langchain-mcp-adapters`（课案 06/07 章同款），
# # 并在每处标注官方新 API 的对应写法，等依赖升级后照着换即可。
# #
# # ### 这一节的运行设计
# #
# # 1. 两个演示服务端的源码**现场生成到临时目录**，不往仓库塞演示文件；
# # 2. Demo 1~3 用 **stdio 传输**拉起服务端子进程 —— 不占端口、不走网络；
# # 3. Demo 4 改用 **HTTP 传输**：因为源文件实测 **stdio 传输 + `create_deep_agent` 会卡死**
# #    （240 秒不返回，而同样的工具交给 `create_agent` 秒回，详见本节末尾的实测结论）；
# # 4. ⚠️ 与源文件最大的不同：源文件把 HTTP 服务端跑在**进程内的后台线程**里（`uvicorn.Server` + `threading`）。
# #    notebook 不能这么干 —— 常驻服务一旦占用内核就再也回不来。所以这里改成
# #    **`subprocess.Popen` 起独立子进程 + 轮询等端口就绪**，并在**本节最后一格**
# #    用 `taskkill /F /T /PID` 连子进程树一起收掉。
#

# # ### 3.1 客户端侧的准备
# #
# # 三件事：
# #
# # 1. **设 `NO_PROXY`**：本机开着 Clash 等系统代理时，`127.0.0.1` 的回环请求会被代理接管
# #    （表现为 `McpError: Session terminated` / 502）。这里是**追加式**写法 ——
# #    环境里已有白名单就把回环补进去，而不是因为 `setdefault` 而整体失效。
# # 2. 导入经典适配器 `MultiServerMCPClient`；
# # 3. 建一个真模型（Demo 1/4 要用它决定调哪个工具）。
#

In [ ]:
import asyncio
import os
import sys

# ⚠️ notebook 专属坑（源 `.py` 脚本里永远遇不到，值得单独记一笔）
# IPython 把 `sys.stderr` 换成了自己的 `OutStream` —— 它**没有真实的 `fileno()`**。
# 而 mcp 的 stdio 客户端会把这个对象当**子进程的 stderr** 交给 `subprocess`；
# Windows 上 subprocess 要拿它的句柄，于是直接抛 `UnsupportedOperation: fileno`。
# 更麻烦的是：mcp 把 `sys.stderr` 写成了**默认参数**（导入时求值绑定），
# 所以必须在 `mcp.client.stdio` 被导入**之前**就换掉，晚一步都无效。
assert "mcp.client.stdio" not in sys.modules, (
    "mcp.client.stdio 已被更早地导入过，errlog 默认值会绑到 IPython 的 OutStream —— 请重启内核后从头运行"
)
_ENV_STDERR_BACKUP = sys.stderr                      # 本节末尾还回去
sys.stderr = open(WORKDIR / "notebook_stderr.log", "w", encoding="utf-8")

for _proxy_key in ("NO_PROXY", "no_proxy"):
    _existing = os.environ.get(_proxy_key, "")
    if "127.0.0.1" not in _existing:
        os.environ[_proxy_key] = (_existing + "," if _existing else "") + "127.0.0.1,localhost"

from langchain_mcp_adapters.client import MultiServerMCPClient

model = init_chat_model(
    model_provider="openai",
    model=settings.model_name,
    api_key=settings.api_key,
    base_url=settings.base_url,
)

print("NO_PROXY =", os.environ["NO_PROXY"])

# # ### 3.2 现场生成两个 stdio 服务端
# #
# # 两个服务端故意都定义了**同名工具 `search`** —— Demo 2 要拿它演示命名冲突。
# #
# # 服务端脚本里有两处刻意为之：
# #
# # | 写法 | 为什么 |
# # |---|---|
# # | `mcp.run(show_banner=False)` | 关掉 FastMCP 的启动 banner（否则会刷满输出，编码还可能错乱） |
# # | `logging.getLogger("fastmcp").setLevel(logging.ERROR)` | 压掉启动 INFO 日志。⚠️ 必须设 **`fastmcp` 这个 logger**：设 root 级别**无效**（fastmcp 用自己的 handler 且不向 root 传播） |
# #
# # 另外注意 `WEATHER_SERVER` 里还注册了一个**资源**（`weather://cities`）和一个
# # `@mcp.prompt`（`docs` 服务端），Demo 3 会取它们。
#

In [ ]:
WEATHER_SERVER = '''
from fastmcp import FastMCP
import logging

logging.getLogger("fastmcp").setLevel(logging.ERROR)   # 压掉它的启动 INFO 日志

mcp = FastMCP("weather-server")


@mcp.tool
def get_weather(city: str) -> str:
    """查询指定城市的天气。"""
    return f"{city}：晴，25℃"


@mcp.tool
def search(query: str) -> str:
    """天气服务里的检索（故意与另一个服务端同名，用来演示命名冲突）。"""
    return f"[天气服务] 命中：{query}"


@mcp.resource("weather://cities")
def supported_cities() -> str:
    """本服务支持的城市清单。"""
    return "北京、上海、广州、深圳"


if __name__ == "__main__":
    # show_banner=False：关掉 FastMCP 的启动 banner（否则会刷满输出，编码还可能错乱）
    mcp.run(show_banner=False)
'''

DOCS_SERVER = '''
from fastmcp import FastMCP
import logging

logging.getLogger("fastmcp").setLevel(logging.ERROR)   # 压掉它的启动 INFO 日志

mcp = FastMCP("docs-server")


@mcp.tool
def search(query: str) -> str:
    """文档服务里的检索（与天气服务同名）。"""
    return f"[文档服务] 命中：{query}"


@mcp.prompt
def explain_topic(topic: str) -> str:
    """生成一段用于讲解某主题的提示词。"""
    return f"请用三句话解释 {topic}，面向初学者。"


if __name__ == "__main__":
    # show_banner=False：关掉 FastMCP 的启动 banner（否则会刷满输出，编码还可能错乱）
    mcp.run(show_banner=False)
'''

In [ ]:
def build_servers(root: Path) -> tuple[Path, Path]:
    """把两个演示服务端写到临时目录，返回脚本路径。"""
    weather = root / "weather_server.py"
    docs = root / "docs_server.py"
    weather.write_text(WEATHER_SERVER, encoding="utf-8")
    docs.write_text(DOCS_SERVER, encoding="utf-8")
    return weather, docs


def stdio_connection(script: Path) -> dict:
    """stdio 连接的配置形状（官方新 API 里叫 transport，经典适配器同理）。"""
    return {"command": sys.executable, "args": [str(script)], "transport": "stdio"}


def tool_result_text(result) -> str:
    """把 MCP 工具的返回值转成人看的文本。

    ⚠️ 实测要点：MCP 工具返回的是 **content block 列表**（形如
    [{'type': 'text', 'text': '...', 'id': ...}]），不是普通字符串 ——
    直接当 str 用会打印出一大坨字典。
    """
    if isinstance(result, list):
        parts = []
        for item in result:
            if isinstance(item, dict):
                parts.append(str(item.get("text", item)))
            else:
                parts.append(str(item))
        return " ".join(parts)
    return str(result)

# # ### 3.3 HTTP MCP 服务端：改成子进程 + 轮询等端口
# #
# # 这是本 notebook 与源文件**唯一的强制改写**，也是模板第 6 节第 5 条的要求。
# #
# # **为什么不能照抄源文件**：源文件用的是 `uvicorn.Server(...)` + 后台线程，在 `.py` 里没问题；
# # 但 notebook 的内核只有一个 —— 常驻服务一旦占住它，后面所有 cell 就永远跑不到了。
# #
# # **改成什么样**：
# #
# # | 步骤 | 做法 |
# # |---|---|
# # | 起服务 | `subprocess.Popen([sys.executable, 服务端脚本])`，把服务端源码写进临时目录 |
# # | 等就绪 | **轮询连端口**（`socket.connect_ex`），而不是 `sleep` 死等一个拍脑袋的秒数 |
# # | 读日志 | `stdout=PIPE` + 一个 daemon 线程把管道读走（⚠️ 见下） |
# # | 关服务 | 本节最后一格 `taskkill /F /T /PID`（`/T` = 连子进程树一起收） |
# #
# # > ⚠️ **为什么必须有个线程去读 stdout**：`stdout=PIPE` 的管道缓冲区只有几 KB，
# # > 子进程写日志写满缓冲区之后会**卡死在写操作上**（经典的 `Popen` 死锁）。
# # > 要么把 stdout 重定向到文件，要么开个线程持续读 —— 这里选后者，顺便把服务端日志收下来做排障。
# #
# # 服务端脚本本身照搬源文件的 HTTP 版本（`mcp.http_app(transport="streamable-http")` + uvicorn），
# # 只是把端口从 `sys.argv[1]` 传进去，避免两边写死不一致。
#

In [ ]:
import socket
import subprocess
import threading

HTTP_PORT = 8110
HTTP_URL = f"http://127.0.0.1:{HTTP_PORT}/mcp"

# 服务端源码：与源文件的进程内版本逐行同构，只是改成由子进程自己跑 uvicorn
HTTP_SERVER = '''
import logging
import sys

import uvicorn
from fastmcp import FastMCP

logging.getLogger("fastmcp").setLevel(logging.ERROR)   # 压掉它的启动 INFO 日志

mcp = FastMCP("weather-http-server")


@mcp.tool
def get_weather(city: str) -> str:
    """查询指定城市的天气。"""
    return f"{city}：晴，25℃"


app = mcp.http_app(transport="streamable-http", path="/mcp")
HTTP_PORT = int(sys.argv[1])          # 端口由父进程传入，两边不会写死得不一致


if __name__ == "__main__":
    server = uvicorn.Server(
        uvicorn.Config(app, host="127.0.0.1", port=HTTP_PORT, log_level="warning")
    )
    server.run()
'''

# # 服务端脚本、探测函数、启动函数、以及一个「删不掉的临时目录不算失败」的清理函数放一起：
#

In [ ]:
def port_ready(port: int) -> bool:
    """探测端口是否已经在监听（比 sleep 死等可靠，也更省时间）。"""
    with socket.socket() as sock:
        sock.settimeout(0.5)
        return sock.connect_ex(("127.0.0.1", port)) == 0


def drain(pipe, sink: list[str]) -> None:
    """后台线程把服务端的 stdout 读走 —— 不读的话管道写满会把服务端卡死。"""
    for line in pipe:
        sink.append(line)


def start_mcp_http_server():
    """用独立子进程起 HTTP MCP 服务端，轮询等端口就绪后返回 Popen 句柄。

    返回句柄是为了让本节的最后一格能把它（连同子进程树）收掉。
    """
    import time as _time

    server_log: list[str] = []
    env = {**os.environ, "PYTHONUTF8": "1", "PYTHONIOENCODING": "utf-8",
           "NO_PROXY": "127.0.0.1,localhost"}
    server = subprocess.Popen(
        [sys.executable, str(root / "mcp_http_server.py"), str(HTTP_PORT)],
        cwd=str(ROOT), env=env,
        stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, encoding="utf-8",
    )
    # 管道必须有人读（否则写满就死锁）；顺便把服务端日志留在内存里，失败时好排障
    thread = threading.Thread(target=drain, args=(server.stdout, server_log), daemon=True)
    thread.start()
    for _ in range(100):          # 最多等 10 秒，轮询到端口真正就绪
        if port_ready(HTTP_PORT):
            return server
        _time.sleep(0.1)
    # 兜底：机器繁忙时冷启动偶发超过 10 秒，再按秒宽限 90 秒
    for _ in range(90):
        if port_ready(HTTP_PORT):
            return server
        _time.sleep(1.0)
    (root / "mcp_http_server.log").write_text("".join(server_log), encoding="utf-8")
    raise RuntimeError(f"HTTP MCP 服务端启动失败（127.0.0.1:{HTTP_PORT} 未监听）")


def remove_temp_dir(path: Path) -> None:
    """删掉临时目录；在 Windows 上删不掉**也不算失败**。

    ⚠️ 实测踩坑：服务端进程树刚被 `taskkill` 掉时，Windows **不会立刻**
    释放它占用的文件句柄。紧接着 `shutil.rmtree` 就会抛
    `PermissionError: [WinError 32] 另一个程序正在使用此文件，进程无法访问。`
    —— 整份演示明明全部跑完了，却崩在最后一行清理上。
    所以这里：先小步重试（等句柄释放），最后一步容忍失败 ——
    清理不掉只是留下一个临时目录，绝不该让演示报错。
    """
    import shutil
    import time as _time

    for _ in range(5):
        try:
            shutil.rmtree(path)
            return
        except FileNotFoundError:
            return
        except OSError:
            _time.sleep(1.0)      # 给 Windows 一点时间释放进程持有的句柄
    shutil.rmtree(path, ignore_errors=True)

# # ### 3.4 准备本次的临时目录
# #
# # 在 `WORKDIR`（同章共享）下面再套一层 `mcp_advanced/` 专属子目录，
# # 把两个 stdio 服务端和 HTTP 服务端脚本都写进去。
#

In [ ]:
MCP_WORK = WORKDIR / "mcp_advanced"
MCP_WORK.mkdir(exist_ok=True)
root = MCP_WORK

(root / "mcp_http_server.py").write_text(HTTP_SERVER, encoding="utf-8")
weather_script, docs_script = build_servers(root)
print(f"演示服务端已生成到临时目录：{root}\n")

# # ### 3.5 Demo 1：连接生命周期 —— 发现工具后即可脱离上下文使用
# #
# # 官方 `connections.mdx` 的核心结论原文：
# #
# # > Discovery happens inside the context, but the tools it returns hold the client,
# # > so they stay callable after the context exits.
# #
# # 即：用 `async with` 打开适配器 → 在上下文内发现工具 → **出了上下文工具照样能调**，
# # 因为工具本身持有客户端（**每次调用时自己开一次会话，用完就放**）。
# #
# # | | 经典适配器（本节） | 官方新 API |
# # |---|---|---|
# # | 拿工具 | `client = MultiServerMCPClient({...})`；`tools = await client.get_tools()` | `async with MCPAdapter(target) as adapter: tools = await adapter.list_tools()` |
# #
# # ⚠️ 两个实测要点：
# # 1. **MCP 工具是异步实现**，必须走 `await agent.ainvoke(...)`（同步 `invoke` 会报错）；
# # 2. 工具在**调用时**各自开会话、用完即放 —— 所以长跑的 agent 不会攥着一条空闲连接。
#

In [ ]:
async def demo_1_lifecycle(weather_script: Path) -> None:
    print("=" * 70)
    print("Demo 1：连接生命周期 —— 发现工具后即可脱离上下文使用")
    print("=" * 70)

    client = MultiServerMCPClient({"weather": stdio_connection(weather_script)})
    tools = await client.get_tools()
    print(f"  发现 {len(tools)} 个工具：{[t.name for t in tools]}")

    # 工具拿在手里后，不需要保持连接上下文（工具每次调用自己开会话）
    agent = create_agent(
        model=model,
        tools=tools,
        system_prompt="你可以调用天气工具。回答简洁，一句话以内。",
    )
    # ⚠️ MCP 工具是**异步**实现：必须走异步入口 ainvoke（同步 invoke 会报错）
    result = await agent.ainvoke({"messages": [{"role": "user", "content": "北京天气怎么样？"}]})
    print(f"  agent 回答：{str(result['messages'][-1].content)[:100]}")
    print(
        "  ↑ 关键点：工具在**发现时**建立连接，在**每次调用时**各自开会话、用完即放 ——\n"
        "    所以长跑的 agent 不会攥着一条空闲连接。这也是官方强调的\n"
        "    「one session per invocation」语义。"
    )


await demo_1_lifecycle(weather_script)

# # ### 3.6 Demo 2：多服务端聚合 + 同名工具的命名空间
# #
# # 官方 `connections.mdx` 专门讲了这件事：用 `MCPConfig` 聚合多个服务端时，
# # **每个工具会自动带上配置键前缀**（`weather_search` / `docs_search`），所以同名工具不会撞车。
# #
# # 本机的经典适配器**不会**自动加前缀 —— 两个 `search` 会重名。
# # 这是迁移到官方新 API 时的一个**行为差异**，本 Demo 演示怎么手工补救。
# #
# # > ✅ **手工加前缀的正确姿势**：按服务端分别取工具（`get_tools(server_name=...)`）再改名。
# # > ❌ **实测踩过的错**：靠工具描述里的关键词猜「这个工具属于谁」——
# # > `docs` 服务端的描述恰好写着「与天气服务同名」，会被误判成天气服务，两个前缀撞在一起。
#

In [ ]:
async def demo_2_multiple_servers(weather_script: Path, docs_script: Path) -> None:
    print("\n" + "=" * 70)
    print("Demo 2：多服务端聚合 —— 同名工具的冲突与手工命名空间")
    print("=" * 70)

    client = MultiServerMCPClient({
        "weather": stdio_connection(weather_script),
        "docs": stdio_connection(docs_script),
    })
    tools = await client.get_tools()
    names = [t.name for t in tools]
    print(f"  两个服务端合计发现 {len(tools)} 个工具：{names}")
    duplicates = {n for n in names if names.count(n) > 1}
    if duplicates:
        print(f"  ⚠️ 重名工具：{sorted(duplicates)} ← 经典适配器不会自动加服务端前缀")
        print("     官方新 API（MCPConfig）会自动加前缀（weather_search / docs_search）；")
        print("     用经典适配器就得自己改名字，否则模型看到两个同名工具会随机选一个。")

    # 手工打前缀的**正确**做法：按服务端分别取工具（get_tools(server_name=...)），
    # 而不是去猜工具描述属于谁 —— 实测踩过：靠描述里的关键词判断会把
    # 「文档服务里的检索（与天气服务同名）」误判成天气服务，两个工具前缀撞在一起。
    renamed = []
    for server_name in ("weather", "docs"):
        for tool in await client.get_tools(server_name=server_name):
            tool.name = f"{server_name}_{tool.name}"
            renamed.append(tool)
    print(f"  按服务端手工加前缀后：{[t.name for t in renamed]}")

    # 分别调用两个同名工具，证明它们确实是两个不同的实现
    for tool in renamed:
        if tool.name.endswith("search"):
            result = await tool.ainvoke({"query": "LangGraph"})
            print(f"    {tool.name} → {tool_result_text(result)}")
    print(
        "  ↑ 多服务端接入的第一道坎不是连接，而是**命名**：\n"
        "    工具名是给模型看的唯一标识，重名等于让模型掷骰子；\n"
        "    正确姿势是按 server_name 分别取工具再加前缀，别靠描述文本猜来源。"
    )


await demo_2_multiple_servers(weather_script, docs_script)

# # ### 3.7 Demo 3：MCP 的三类原语在客户端侧怎么取
# #
# # MCP 有三个原语，课案 `05_mcp` 章讲了服务端怎么定义，这里看客户端怎么取：
# #
# # | 原语 | 客户端取法（经典适配器） | 返回值形态（实测） |
# # |---|---|---|
# # | tools | `await client.get_tools()` | LangChain 工具，能直接交给 agent |
# # | resources | `await client.get_resources(server)` | `Blob`（含 `uri` / `mimetype` / `data`），要自己喂给模型 |
# # | prompts | `await client.get_prompt(server, name, arguments=...)` | **消息列表**（`[HumanMessage(...)]`），不是字符串 |
# #
# # 官方新 API 里资源与提示词同样由 `MCPAdapter` 暴露。
# # 两处都套了 `try/except`，因为原语的可用性取决于服务端实现，不该让演示整体崩掉。
#

In [ ]:
async def demo_3_resources_and_prompts(weather_script: Path, docs_script: Path) -> None:
    print("\n" + "=" * 70)
    print("Demo 3：资源与提示词 —— 客户端侧的取法")
    print("=" * 70)

    client = MultiServerMCPClient({
        "weather": stdio_connection(weather_script),
        "docs": stdio_connection(docs_script),
    })

    # 资源：只读数据（这里用自定义 URI 方案 weather://cities）
    try:
        resources = await client.get_resources("weather")
        print(f"  weather 服务端的资源（{len(resources)} 个）：")
        for item in resources:
            print(f"    uri={getattr(item, 'uri', item)}")
    except Exception as exc:  # noqa: BLE001
        print(f"  取资源失败：{type(exc).__name__}: {str(exc)[:120]}")

    # 提示词：服务端预置的提示模板
    try:
        prompt = await client.get_prompt("docs", "explain_topic", arguments={"topic": "MCP 协议"})
        text = getattr(prompt, "messages", prompt)
        print(f"  docs 服务端的提示词 explain_topic → {str(text)[:120]}")
    except Exception as exc:  # noqa: BLE001
        print(f"  取提示词失败：{type(exc).__name__}: {str(exc)[:120]}")

    print(
        "  ↑ 三类原语的客户端取法（经典适配器）：\n"
        "    tools     → await client.get_tools()          （能直接交给 agent）\n"
        "    resources → await client.get_resources(server)（只读数据，要自己喂给模型）\n"
        "    prompts   → await client.get_prompt(server, name, arguments=...)\n"
        "    官方新 API 里资源与提示词同样由 MCPAdapter 暴露。"
    )


await demo_3_resources_and_prompts(weather_script, docs_script)

# # ### 3.8 Demo 4：把 MCP 工具交给 DeepAgents（覆盖缺口表 DeepAgents 第 8 项）
# #
# # 官方 `deepagents/tools.mdx#mcp-tools` 的原文结论很简单：
# #
# # > Load tools from any MCP server and pass them directly to `create_deep_agent`.
# #
# # 也就是说 **DeepAgents 自己没有任何 MCP 适配层**（本地实测：`deepagents` 包里
# # 找不到任何 MCP 符号），它只是接收 LangChain 工具 —— 所以 Demo 1 拿到的工具
# # **原样塞进去即可**，同时白拿 DeepAgents 的文件系统、子代理、上下文压缩等能力。
# #
# # ⚠️ 但**传输方式有讲究**：源文件实测 `stdio` 传输 + `create_deep_agent` 会**卡死**，
# # 换成 `streamable_http` 立刻正常 —— 所以这一格先把 HTTP 服务端拉起来（子进程）。
#

In [ ]:
print(f"\n（Demo 4 启动 HTTP MCP 服务端（独立子进程，端口 {HTTP_PORT}）：{HTTP_URL}）")
server = start_mcp_http_server()
print("   服务端已就绪，PID =", server.pid)

# # 服务端就绪后跑 Demo 4。外面套一层 `asyncio.wait_for` 是刻意的兜底：
# # 源文件记录过 stdio 会卡 240 秒不返回，虽然这里已经改用 HTTP，
# # 仍用墙钟超时把「万一又卡住」的情况兜住，别让整份 notebook 拖死在这一格。
#

In [ ]:
async def demo_4_deepagents() -> None:
    print("\n" + "=" * 70)
    print("Demo 4：MCP 工具 → create_deep_agent（DeepAgents 第 8 项）")
    print("=" * 70)

    from deepagents import create_deep_agent

    client = MultiServerMCPClient({"weather": {"url": HTTP_URL, "transport": "streamable_http"}})
    tools = await client.get_tools()

    agent = create_deep_agent(
        model=model,
        tools=tools,          # ← MCP 工具直接传进去，和普通工具没有区别
        system_prompt="你可以调用 MCP 天气工具。回答一句话即可。",
    )
    result = await agent.ainvoke(
        {"messages": [{"role": "user", "content": "上海天气如何？"}]},
        config={"recursion_limit": 20},   # 兜底：防止意外的长循环把演示拖死
    )
    print(f"  发现 {len(tools)} 个 MCP 工具：{[t.name for t in tools]}")
    print(f"  deep agent 回答：{str(result['messages'][-1].content)[:100]}")
    print(
        "  ↑ DeepAgents 不重复造 MCP 适配层：**工具就是工具**。\n"
        "    所以「MCP + 深度智能体」的组合成本极低 —— 接完协议，文件系统/子代理/\n"
        "    上下文压缩全部白拿（课案 03_deepagents 章讲的那些能力）。\n"
        "    ⚠️ 但传输方式有讲究：本 Demo 走 HTTP 而不是 stdio，原因见文末实测结论第 2 条。"
    )


async def main() -> None:
    """Demo 4 的异步入口（源文件里这一段也在 `main()` 内）。"""
    await demo_4_deepagents()


await asyncio.wait_for(main(), timeout=300)

# # ### 3.9 收尾：关掉 HTTP MCP 服务端
# #
# # **这是本节（也是整个 notebook 唯一会起常驻服务的地方）的最后一格。**
# #
# # 为什么要用 `taskkill /F /T /PID`：
# #
# # | 参数 | 作用 |
# # |---|---|
# # | `/F` | 强制结束（子进程在跑事件循环，不给它优雅退出的机会） |
# # | `/T` | **连子进程树一起收** —— 光杀父进程会留下孤儿 uvicorn 吊着端口 |
# # | `/PID` | 指定父进程 PID |
# #
# # `try/finally` 是刻意的：`taskkill` 万一失败或被 Ctrl-C 打断，
# # **临时目录的回收也必须走完**（否则下次跑还会看到残留的旧服务端脚本）。
# # 而 `remove_temp_dir` 本身对 Windows 的 `WinError 32` 重试 + 容忍失败。
#

In [ ]:
try:
    if server.poll() is None:
        subprocess.run(["taskkill", "/F", "/T", "/PID", str(server.pid)],
                       stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL, check=False)
finally:
    remove_temp_dir(root)

print("HTTP MCP 服务端子进程 poll =", server.poll(), "（非 None 即已退出）")
print("临时目录还在吗？", root.exists())

# 把 sys.stderr 还给 IPython。注意**不关**那个文件对象：mcp 侧记下的默认值还指着它，
# 保持打开才能让你重跑第 3 节时 stdio 传输依然可用。
sys.stderr = _ENV_STDERR_BACKUP

print("\n全部 Demo 执行完毕（临时目录已清理，HTTP 服务端已关闭）。")

# # ### 3.10 本节实测结论与踩坑
# #
# # **实测环境**：`langchain-mcp-adapters 0.3.2` / `fastmcp 3.4.7` / `deepagents 0.7.13`。
# #
# # 1. 官方新 API `langchain.mcp.MCPAdapter` **不可用**：导入即报
# #    `No module named 'fastmcp.client.group'`（要 fastmcp 4.x，本机 3.4.7）；
# # 2. 经典适配器 `MultiServerMCPClient` 可用：`await client.get_tools()` 拿到工具，
# #    `await tool.ainvoke({...})` 调用成功；
# # 3. **MCP 工具的返回值是 content block 列表**（`[{'type': 'text', 'text': ...}]`），
# #    不是字符串 —— 直接 `str()` 会打出一坨字典（本节提供了 `tool_result_text` 转换）；
# # 4. MCP 工具是**异步**实现，agent 必须走 `ainvoke`；
# # 5. `await client.get_tools(server_name="docs")` 可按服务端分别取工具 ——
# #    这是「手工加命名空间前缀」的正确姿势（靠工具描述猜来源会误判，实测踩过）；
# # 6. `get_resources()` 返回 `Blob`，`get_prompt()` 返回**消息列表**，都不是字符串；
# # 7. `deepagents` 包里**没有任何 MCP 符号**：官方路线就是把工具传进去；
# # 8. ★ **stdio 传输 + `create_deep_agent` 会卡死**：同一个 stdio MCP 工具交给
# #    `create_agent` 秒回（Demo 1），交给 `create_deep_agent` 则 **240 秒不返回**
# #    （起两次实测均如此）；换成 `streamable_http` 立刻正常返回。
# #    **结论：DeepAgents 接 MCP 时用 HTTP 传输**（课案 `05_mcp/07` 用的正是 HTTP，结论一致）。
# #
# # 本节**未收录**（官方还有、这里没做的）：
# #
# # - 官方新 API 的 `MCPConfig` 自动前缀与 `ClientGroup`（每服务端独立连接、可混协议代际）——
# #   都要 fastmcp 4.x，本机依赖不具备，接口写法已在上面标注；
# # - OAuth / bearer 认证的**客户端侧**（`mcp/auth.mdx`）—— 课案 `05_mcp` 已用 JWT 讲透同类机制，
# #   且那一套要起两个常驻服务；
# # - 部署侧共享连接池与缓存（`connections.mdx` 的 Scale a deployment）—— 属服务化部署话题。
#

# # ## 4. RAG 知识库（来自 `24_RAG知识库_官方补充.py`）
# #
# # 官方 `knowledge-base.mdx` 讲的是「让 agent 能查自己的知识」的标准流程：
# #
# # ```text
# # 加载文档 → 切分 → 向量化 → 存向量库 → 检索 →（可选）重排序 → 交给模型作答
# # ```
# #
# # ### 与本机配置的对应关系（这是本节与前三节最大的不同）
# #
# # | 角色 | 模型 | 端点 | 实测 |
# # |---|---|---|---|
# # | 向量化 | `BAAI/bge-m3` | SiliconFlow | **1024 维**，单次 0.12~0.3 秒 |
# # | 精排 | `BAAI/bge-reranker-v2-m3` | SiliconFlow | 0.2~0.25 秒 |
# # | 作答 | `.env` 里的 `MODEL_NAME` | 模型网关 | 见 Demo 4/5 的真实耗时与 token |
# #
# # 三者都走 `from config import settings`（课案约定：配置集中在 `.env` + `config.py`）。
# #
# # ### 为什么必须有「重排序」这一步（本节的核心教学点）
# #
# # | | 向量检索 | 重排序 |
# # |---|---|---|
# # | 结构 | **双塔**：query 与文档各自编码成向量，再算距离 | **交叉编码**：query 和文档拼在一起送进模型逐对打分 |
# # | 优点 | 快，能扫百万级 | 准得多（query 和文档「见过面」） |
# # | 缺点 | 精度有限（两者从没见过面） | 慢，只能处理少量候选 |
# # | 用在 | **召回**（负责不漏） | **精排**（负责排序准） |
# #
# # 生产标配：**向量召回 top-20 → rerank 精排 top-5 → 交给模型**。
# # Demo 3 会把「召回顺序」和「精排顺序」并排打印出来，差异一眼可见。
#

# # ### 4.1 模型与向量化组件
# #
# # 两个构造里各有一个**必须注意的开关**：
# #
# # - `OpenAIEmbeddings` 连第三方 OpenAI 兼容端点时，必须 `check_embedding_ctx_length=False`
# #   —— 否则它会按 OpenAI 的 tiktoken 规则先切分文本，遇到中文/长文本容易报错；
# # - `request_timeout=60` 是**显式超时**：端点慢时快速失败，而不是无限挂住（实测踩过：没设超时时整跑会卡死）。
#

In [ ]:
import warnings

# 本机 jupyter / ipywidgets 的版本搭配会让 tqdm 在导入时抛一条
# 「IProgress not found. Please update jupyter and ipywidgets」警告 ——
# 与本课内容无关，但会稳定地污染每一格的输出，所以显式静音。
warnings.filterwarnings("ignore", message=".*IProgress not found.*")

import json
import tempfile
import time
import urllib.request

from langchain.tools import tool
from langchain_core.documents import Document
from langchain_core.vectorstores import InMemoryVectorStore
from langchain_openai import OpenAIEmbeddings
from langchain_text_splitters import RecursiveCharacterTextSplitter

llm = init_chat_model(
    model_provider="openai",
    model=settings.model_name,
    api_key=settings.api_key,
    base_url=settings.base_url,
    max_retries=0,
)

embeddings = OpenAIEmbeddings(
    model=settings.embedding.model,
    api_key=settings.embedding.api_key,
    base_url=settings.embedding.base_url,
    check_embedding_ctx_length=False,   # 兼容第三方端点的必要开关（见文末踩坑）
    # 显式超时：端点慢时快速失败，而不是无限挂住（实测踩过：没设超时时整跑会卡死）
    request_timeout=60,
    max_retries=1,
)

# # ### 4.2 知识库素材
# #
# # 真实项目里是 PDF / 网页 / 数据库，这里现场生成三篇小文档（LangGraph、RAG、运维值班手册），
# # 够用就行。`QUERIES` 是后面所有 Demo 共用的两个问题 ——
# # 一个问 RAG 自身（考「重排序」那段），一个问 LangGraph（考跨文档召回）。
#

In [ ]:
DOCS: dict[str, str] = {
    "langgraph.md": (
        "# LangGraph 基础\n\n"
        "LangGraph 用节点和边描述流程：节点是普通函数，边决定执行顺序。\n\n"
        "持久化由 checkpointer 负责，thread_id 用来区分不同会话，支持中断恢复与时间旅行。\n\n"
        "状态用 TypedDict 定义，列表字段要配 reducer（如 operator.add），否则后写会覆盖先写。\n"
    ),
    "rag.md": (
        "# RAG 检索增强\n\n"
        "RAG 的标准流程是：加载 → 切分 → 向量化 → 检索 → 重排序 → 交给模型作答。\n\n"
        "向量检索属于双塔结构，速度快但精度有限；重排序用交叉编码器逐对打分，精度高但更慢，"
        "因此只精排少量候选。\n\n"
        "常见坑：切分粒度太大导致检索命中但答不准；切分太小则上下文碎片化。\n"
    ),
    "ops.md": (
        "# 运维值班手册\n\n"
        "值班期间每两小时巡检一次机房温度，超过 28 摄氏度需要报备。\n\n"
        "重启服务前必须先确认没有正在执行的批处理任务，否则可能造成数据不一致。\n"
    ),
}

QUERIES = [
    "为什么要做重排序？它和向量检索有什么区别？",
    "LangGraph 的会话是怎么区分和恢复的？",
]

# # ### 4.3 建库三步：加载 → 切分 → 向量化
# #
# # 三步里**最容易出问题的是切分**：粒度太大 → 检索命中但答不准；太小 → 上下文碎片化。
# #
# # 中文场景必须**显式给分隔符** —— 默认分隔符里没有中文标点，
# # 不写 `"。"` / `"；"` / `"，"` 就会按英文标点切，切出来的块语义不完整。
# #
# # 向量库用 `InMemoryVectorStore`（生产用 Milvus，见课案的 `RAG/` 项目）：
# # 演示重点在**流程**，不在存储引擎。
#

In [ ]:
def build_knowledge_base(root: Path) -> tuple[InMemoryVectorStore, list[Document]]:
    """把素材写到磁盘 → 加载 → 切分 → 向量化入库，返回（向量库, 切片列表）。"""
    for name, content in DOCS.items():
        (root / name).write_text(content, encoding="utf-8")

    # ① 加载：这里直接读文件（生产常用 TextLoader / PyPDFLoader / 网页加载器）
    raw_documents = [
        Document(page_content=(root / name).read_text(encoding="utf-8"), metadata={"source": name})
        for name in sorted(DOCS)
    ]

    # ② 切分：中文场景建议显式给分隔符（默认分隔符里没有中文标点）
    splitter = RecursiveCharacterTextSplitter(
        chunk_size=120,
        chunk_overlap=20,
        separators=["\n\n", "\n", "。", "；", "，", " ", ""],
    )
    chunks = splitter.split_documents(raw_documents)

    # ③ 向量化入库：InMemoryVectorStore 适合演示（生产用 Milvus，见课案 RAG 项目）
    store = InMemoryVectorStore(embedding=embeddings)
    store.add_documents(chunks)
    return store, chunks

# # ### 4.4 重排序组件：LangChain 没有内置 SiliconFlow 的 rerank，自己包一层
# #
# # `SiliconFlowReranker` 只做一件事：POST 到 `/rerank`。
# # 请求体是 `{model, query, documents, top_n}` —— 与 OpenAI 的 `/embeddings` 不同构，
# # 但同为 JSON + Bearer 鉴权。
# #
# # ⚠️ **为什么显式不走代理**：本机系统代理会接管部分请求，
# # 这里用 `urllib.request.build_opener(ProxyHandler({}))` 明确禁用（内网课案环境同款处理）。
# #
# # ⚠️ **分数怎么读（两次实测对照得出的结论）**：
# #
# # - 分数是 sigmoid 后的 `[0, 1]` 值，但**绝对高低取决于「语料里有没有真答案」**：
# #   本文件有合适文档时第一名 0.95、第二名 0.003（差 300 倍）；
# #   而探针里用一句和语料无关的查询时，最高分只有 0.0288。
# # - 所以**别用固定绝对阈值过滤**（`.env` 里 `RERANK_RELEVANCE_P=0.65` 在这种尺度下会
# #   把「没有答案」的场景全部滤掉，也让「有答案」的场景显得过松）；
# #   **看 top1 与 top2 的相对差距**判断是否命中，阈值按自己的语料重标定。
# #
# # 数据合规提醒：**候选文本会发到云端打分**。本地部署的交叉编码器
# # （如 `sentence-transformers` + bge-reranker）数据不出机器，但要多一份显存/运维成本。
#

In [ ]:
class SiliconFlowReranker:
    """调用 SiliconFlow `/rerank` 的精排组件（**候选文本会发到云端打分**，注意数据合规）。

    对比：本地部署的交叉编码器（如 sentence-transformers + bge-reranker）数据不出机器，
    但要多一份显存/内存与运维成本；本文件用云端 API，换来零部署。

    实测注意（很重要，两次实测对照得出）：
        · 分数是 sigmoid 后的 [0, 1] 值，但**绝对高低取决于"语料里有没有真答案"**：
          本文件有合适文档时第一名 0.95、第二名 0.003（差 300 倍）；
          而探针里用一句和语料无关的查询时，最高分只有 0.0288。
        · 所以别用固定绝对阈值过滤（.env 里 RERANK_RELEVANCE_P=0.65 在这种尺度下会
          把「没有答案」的场景全部滤掉，也让「有答案」的场景显得过松）；
          **看 top1 与 top2 的相对差距**判断是否命中，阈值按自己的语料重标定。
        · 请求体是 `{model, query, documents, top_n}`，与 OpenAI 的 /embeddings 不同构，
          但同为 JSON + Bearer 鉴权。
    """

    def __init__(self, model: str, api_key: str, base_url: str, timeout: int = 60) -> None:
        self.model = model
        self.api_key = api_key
        self.url = base_url.rstrip("/") + "/rerank"
        self.timeout = timeout
        # 本机系统代理会接管部分请求，这里显式不走代理（内网课案环境同款处理）
        self._opener = urllib.request.build_opener(urllib.request.ProxyHandler({}))

    def rerank(self, query: str, candidates: list[str], top_n: int = 5) -> list[dict]:
        """对候选文本重排，返回 [{index, score}]（按分数从高到低）。"""
        payload = {
            "model": self.model,
            "query": query,
            "documents": candidates,
            "top_n": min(top_n, len(candidates)),
        }
        request = urllib.request.Request(
            self.url,
            data=json.dumps(payload).encode("utf-8"),
            headers={
                "Content-Type": "application/json",
                "Authorization": f"Bearer {self.api_key}",
            },
        )
        with self._opener.open(request, timeout=self.timeout) as response:
            body = json.loads(response.read().decode("utf-8"))
        return [
            {"index": item["index"], "score": item.get("relevance_score", 0.0)}
            for item in body.get("results", [])
        ]


reranker = SiliconFlowReranker(
    model=settings.rerank.model,
    api_key=settings.rerank.api_key,
    base_url=settings.rerank.base_url,
)

# # ### 4.5 建库（并解释「为什么要用 `with`」）
# #
# # 用 `tempfile.TemporaryDirectory` 开一个临时目录放这三篇文档：
# # `with` 块退出时目录会被自动清理。
# #
# # > **为什么清理了也没关系**：真正要留下来的是**内存里的向量库 `store`**（和切片 `chunks`）。
# # > 原始文件只是**输入**，入库之后就不再被读了 —— 后面的检索、重排、作答全部只碰 `store`。
# # > 这也是 RAG 的一个工程事实：**源文档可以丢，向量库不能丢。**
#

In [ ]:
with tempfile.TemporaryDirectory(prefix="rag_demo_") as tmp:
    root = Path(tmp)
    print(f"演示知识库目录：{root}\n")
    store, chunks = build_knowledge_base(root)

# # ### 4.6 Demo 1：构建结果 —— 切了多少片、向量多少维
# #
# # 除了打印切片数量与前几片内容，这一格还**单独验证一次 embedding 端点**并拿到维度 ——
# # 换模型时这是最快的自检：bge-m3 应当是 **1024 维**，维度对不上说明端点和模型对不上号。
#

In [ ]:
def demo_1_build(chunks: list[Document]) -> None:
    print("=" * 70)
    print("Demo 1：构建知识库 —— 加载 → 切分 → 向量化")
    print("=" * 70)

    print(f"  原始文档：{len(DOCS)} 篇，共 {sum(len(c) for c in DOCS.values())} 字符")
    print(f"  切分参数：chunk_size=120 / overlap=20（中文分隔符已显式指定）")
    print(f"  切片数量：{len(chunks)}")
    for index, chunk in enumerate(chunks[:3], start=1):
        source = chunk.metadata.get("source", "?")
        print(f"    切片 {index}（{source}）：{chunk.page_content[:40].replace(chr(10), ' ')}…")

    # 单独验证一次 embedding 端点，顺便拿到维度（bge-m3 = 1024 维）
    started = time.time()
    vector = embeddings.embed_query("测试向量维度")
    print(f"  向量维度：{len(vector)}（模型 {settings.embedding.model}），"
          f"单次向量化耗时 {time.time()-started:.2f}s")
    print(
        "  ↑ 三步里最容易出问题的是**切分**：粒度太大 → 检索命中但答不准；\n"
        "    太小 → 上下文碎片化。中文记得把「。」「；」写进 separators，\n"
        "    否则默认分隔符会按英文标点切，切出来的块语义不完整。"
    )


demo_1_build(chunks)

# # ### 4.7 Demo 2：向量检索 —— 只看召回顺序
# #
# # `similarity_search_with_score(query, k=5)` 返回 `(Document, score)` 列表。
# #
# # ⚠️ `InMemoryVectorStore` 返回的是**相似度**（越大越像）；
# # 注意它和 rerank 的分数**不是一个尺度**，别混着比 —— Demo 3 会把两者并排打印，这一点会很直观。
#

In [ ]:
def demo_2_vector_search(store: InMemoryVectorStore) -> None:
    print("\n" + "=" * 70)
    print("Demo 2：向量检索 —— 召回的原始顺序")
    print("=" * 70)

    query = QUERIES[0]
    started = time.time()
    hits = store.similarity_search_with_score(query, k=5)
    print(f"  查询：{query}")
    print(f"  召回 {len(hits)} 条，耗时 {time.time()-started:.2f}s（含一次 query 向量化）")
    for rank, (document, score) in enumerate(hits, start=1):
        text = document.page_content.replace("\n", " ")[:46]
        print(f"    {rank}. 相似度 {score:.4f}｜{document.metadata.get('source')}｜{text}…")
    print(
        "  ↑ InMemoryVectorStore 返回的是**相似度**（越大越像）；\n"
        "    注意它和 rerank 的分数**不是一个尺度**，别混着比。"
    )


demo_2_vector_search(store)

# # ### 4.8 Demo 3：召回顺序 vs 精排顺序（本节的核心对照）
# #
# # 两个查询各跑一遍同一套流程：
# #
# # 1. 先按向量召回 `k=5`，打印**召回顺序**；
# # 2. 再把这 5 条候选交给 rerank 精排 `top_n=5`，打印**精排顺序与分数**；
# # 3. 比较 `ranked[0]["index"]` 是不是 0 —— 不是，就说明**交叉编码把更相关的候选提到了前面**；
# # 4. 打印 top1 与 top2 的**相对差距**，并解释怎么用它判断「语料里到底有没有答案」。
#

In [ ]:
def demo_3_rerank(store: InMemoryVectorStore) -> None:
    print("\n" + "=" * 70)
    print("Demo 3：重排序 —— 向量召回顺序 vs 交叉编码精排顺序")
    print("=" * 70)

    for query in QUERIES:
        print(f"\n  查询：{query}")
        candidates = [document.page_content for document in store.similarity_search(query, k=5)]
        print("    ① 向量召回顺序：")
        for rank, text in enumerate(candidates, start=1):
            print(f"       {rank}. {text.replace(chr(10), ' ')[:44]}…")

        started = time.time()
        ranked = reranker.rerank(query, candidates, top_n=5)
        print(f"    ② rerank 精排顺序（{time.time()-started:.2f}s，模型 {settings.rerank.model}）：")
        for rank, item in enumerate(ranked, start=1):
            text = candidates[item["index"]].replace("\n", " ")[:44]
            print(f"       {rank}. 分数 {item['score']:.6f}｜{text}…")

        if ranked and ranked[0]["index"] != 0:
            print("    ✔ 顺序发生了变化：交叉编码把更相关的候选提到了前面")
        else:
            print("    （本次顺序未变：召回的第一条本身就是最相关的）")
        if len(ranked) >= 2:
            top, second = ranked[0]["score"], ranked[1]["score"]
            gap_desc = f"相差 {top / second:.0f} 倍" if second > 0 else "第二名分数为 0（差距无穷大）"
            print(f"    ★ 第一名与第二名差距：{top:.4f} vs {second:.6f}（{gap_desc}）"
                  f" → 说明语料里确实有答案，且 top1 明显更相关")
            print("      （反过来：若所有候选分数都在同一量级且很低，说明**语料里没有答案**，"
                  "这时候应该让模型直说不知道，而不是硬塞上下文）")
    print(
        "\n  ↑ 这就是「召回 + 精排」两级检索的意义：\n"
        "    召回负责**不漏**（双塔快，扫全库），精排负责**排序准**（交叉编码逐对看）。\n"
        "    生产里 rerank 只处理召回的几十条，成本可控。"
    )


demo_3_rerank(store)

# # ### 4.9 Demo 4：把精排后的上下文交给模型作答
# #
# # 流程：召回 5 条 → 精排取 **top-2** → 拼成 `【资料 N】…` 塞进 prompt → 让模型作答。
# #
# # ⚠️ prompt 里的约束是**故意的**：
# #
# # ```text
# # 你是严谨的助手。只依据【资料】回答，资料没写的内容就说不知道，不要编造。
# # ```
# #
# # RAG 最常见的翻车方式就是**模型拿常识补齐了资料里没有的内容**，
# # 所以这条约束（以及引用来源）要**写死在提示词里**。
# #
# # 这一格还会打印真实耗时与 token 用量 —— 换模型/换端点时，这是最直接的体感指标。
#

In [ ]:
def demo_4_answer_with_context(store: InMemoryVectorStore) -> None:
    print("\n" + "=" * 70)
    print("Demo 4：把精排后的上下文交给模型作答")
    print("=" * 70)

    query = QUERIES[0]
    candidates = [document.page_content for document in store.similarity_search(query, k=5)]
    ranked = reranker.rerank(query, candidates, top_n=2)
    context = "\n\n".join(f"【资料 {i+1}】{candidates[item['index']]}" for i, item in enumerate(ranked))

    prompt = (
        "你是严谨的助手。只依据【资料】回答，资料没写的内容就说不知道，不要编造。\n\n"
        f"{context}\n\n问题：{query}"
    )
    started = time.time()
    response = llm.invoke(prompt)
    usage = getattr(response, "usage_metadata", None) or {}
    print(f"  送入模型的资料条数：{len(ranked)}（来自召回 {len(candidates)} 条）")
    print(f"  回答（{time.time()-started:.1f}s，{usage.get('total_tokens')} tokens）：")
    print(f"    {str(response.content)[:300]}")
    print(
        "  ↑ 注意 prompt 里的约束「只依据资料、没有就说不知道」——\n"
        "    RAG 最常见的翻车方式就是模型拿常识补齐了资料里没有的内容，\n"
        "    所以这条约束（以及引用来源）要写死在提示词里。"
    )


demo_4_answer_with_context(store)

# # ### 4.10 Demo 5：把检索做成工具 —— agentic RAG
# #
# # 出处是 **`deepagents/retrieval.mdx` 的 "Agentic RAG" 一节**，注意**不是** `knowledge-base.mdx`
# # —— 那篇止于建库与检索，全文没有 agent 环节。
# #
# # 做法就是把上面的「召回 + 精排」合成一个 `@tool` 交给 `create_agent`，
# # 让模型自己决定：**要不要查、查几次、查询词怎么写**。
# #
# # ⚠️ 工具返回的内容要**限长**（这里取 `top_2`），否则多次调用会把上下文撑爆
# # （课案 `03_deepagents/14` 讲的上下文卸载机制同理）。
# #
# # 这一格的观察点是 `called` 列表（模型实际调用的工具名）：
# # 问题是「机房温度 + 有没有讲重排序」两件事，**通常会查两次** ——
# # 但那是**模型决策**，不是必然，别把它写成断言。
#

In [ ]:
def demo_5_agentic_rag(store: InMemoryVectorStore) -> None:
    print("\n" + "=" * 70)
    print("Demo 5：agentic RAG —— 把检索+精排做成工具")
    print("=" * 70)

    @tool
    def search_knowledge_base(query: str) -> str:
        """在内部知识库里检索（自动做向量召回 + 交叉编码精排），返回最相关的若干段资料。"""
        candidates = [document.page_content for document in store.similarity_search(query, k=5)]
        ranked = reranker.rerank(query, candidates, top_n=2)
        return "\n---\n".join(candidates[item["index"]] for item in ranked) or "（没有检索到资料）"

    agent = create_agent(
        model=llm,
        tools=[search_knowledge_base],
        system_prompt=(
            "你是知识库助手。回答前**必须先调用 search_knowledge_base** 查资料；"
            "资料里没有的内容就直说不知道。回答控制在三句话内。"
        ),
    )
    question = "值班的时候机房温度超了怎么办？还有，知识库里有没有讲重排序的？"
    result = agent.invoke({"messages": [{"role": "user", "content": question}]})

    called = [
        call["name"]
        for message in result["messages"]
        for call in (getattr(message, "tool_calls", None) or [])
    ]
    print(f"  提问：{question}")
    print(f"  模型调用的工具：{called}")
    print(f"  回答：{str(result['messages'][-1].content)[:260]}")
    search_calls = [name for name in called if name == "search_knowledge_base"]
    print(
        f"  ↑ 与 Demo 4 的区别：**查不查、查几次由模型决定**（本次它调了 {len(search_calls)} 次；\n"
        "    问题里有两件事时通常会查两次，但这是**模型决策**，不是必然）。\n"
        "    这就是 agentic RAG：检索成了 agent 的能力，而不是流水线里固定的一步。"
    )


demo_5_agentic_rag(store)

In [ ]:
print("\n全部 Demo 执行完毕（临时目录已清理）。")

# # ### 4.11 本节实测结论与踩坑
# #
# # **实测环境**：`BAAI/bge-m3` + `BAAI/bge-reranker-v2-m3` @ SiliconFlow，作答走 `.env` 的模型网关。
# #
# # 1. embedding 实测 **1024 维**，单次 0.12~0.3 秒，返回 `usage`；rerank 0.2~0.25 秒；
# # 2. **rerank 分数要按相对差距解读**：有答案时 top1 ≈ 0.95 / top2 ≈ 0.003（差 300 倍），
# #    无答案时最高分只有 0.0288 —— 用「top1 与 top2 的差距」判断命中，
# #    而不是固定阈值（`.env` 里 `RERANK_RELEVANCE_P=0.65` 在这种尺度下会误判）；
# # 3. 与课案 `RAG/` 项目（Milvus + 双路召回 + ES）的分工：
# #    本节的目的是把**官方 LangChain 侧的 RAG 写法**讲清楚，用的是
# #    `InMemoryVectorStore` + 单路向量召回；两者模型配置**共用根目录 `.env`**
# #    （`EMBEDDING_*` / `RERANK_*` / `LLM_*`）。混合检索（BM25 + 向量）见
# #    `RAG/retrieval/keyword_retrieval.py` 与 `vector_retrieval.py`；
# #    **本节的 rerank 组件可以直接替换那边的手写实现**。
# #
# # **踩坑清单**：
# #
# # | # | 坑 | 解法 |
# # |---|---|---|
# # | A | `OpenAIEmbeddings` 连第三方兼容端点时按 OpenAI 的 tiktoken 规则切文本，中文/长文本容易报错 | 必须 `check_embedding_ctx_length=False` |
# # | B | 端点慢时无限挂住 | 显式给 `request_timeout`（本节 60 秒） |
# # | C | rerank 分数被当概率用 | **只按排名用**，不同厂商尺度差异极大 |
# # | D | 中文切分被按英文标点切 | 自定义 `separators`，把 `。` `；` `，` 写进去 |
# # | E | 检索结果没来源，答案无法溯源 | 带上 `metadata["source"]` —— 否则排障时不知道是「没检索到」还是「检索到了但模型没用」 |
# # | F | 工具化的检索把上下文撑爆 | 返回内容**限长**（本节取 top-2） |
#

# # ## 小结
# #
# # | 节 | 一句话记住 |
# # |---|---|
# # | 1. LCEL 管道 | `|` 是「上一个的输出 = 下一个的输入」，`invoke`/`stream`/`batch` 同一对象三种用法；但**它已不是官方主线，新编排用 LangGraph** |
# # | 2. 测试与护栏 | 单测靠**假模型**（离线可断言）；评估靠**轨迹**（断言流程而不是措辞）；护栏靠**短路**（在花钱之前拦住）；上下文靠 **ToolRuntime 注入** |
# # | 3. MCP 进阶 | 工具**发现时连、调用时开会话**；多服务端第一道坎是**命名**；MCP 工具返回值是 **content block 列表**；DeepAgents 接 MCP 要用 **HTTP 传输** |
# # | 4. RAG 知识库 | **召回负责不漏、精排负责排序准**；rerank 分数看**相对差距**；把检索做成工具就得到 **agentic RAG** |
# #
# # 四节串起来就是一条完整链路：**写得出来（1）→ 证明得了（2）→ 接得上外部（3）→ 查得到自己的资料（4）**。
#

# # ## 常见坑
# #
# # **第 1 节**
# #
# # 1. **别把 LCEL 当官方主线**：官方新文档零命中 "LCEL"，新编排逻辑请用 LangGraph 的 `StateGraph`。
# # 2. **管道里的普通函数是合法的**：`| (lambda text: ...)` 能跑，说明管道只是「输出喂输入」的约定。
# #
# # **第 2 节**
# #
# # 3. **`GenericFakeChatModel` 的 `messages` 是一次性迭代器**：剧本用完再 `invoke` 会 `StopIteration`
# #    —— 每个用例都要**重建模型实例**。
# # 4. **`subset` / `superset` 极易记反**：记住方向 —— `subset` 是**实际不许比参考多**，
# #    `superset` 是**实际允许比参考多**。
# # 5. **别把工具参数命名成 `runtime` / `config`**：它们是保留字，不会出现在给模型的 schema 里。
# # 6. **不传 `context` 不会在编译期报错**，而是在工具执行时抛 `AttributeError` 并**中断整个运行**
# #    （`ToolNode` 默认只把 `ToolInvocationError` 转成消息，其余一律往外抛）。
# #    生产建议在调用封装层统一注入，避免漏传。
# # 7. **轨迹断言别写成「回复文本全等」**：模型措辞天然会变，那是最脆的测试；断言流程与状态。
# #
# # **第 3 节**
# #
# # 8. **MCP 工具返回值是 content block 列表**，喂给模型前最好自己转文本（本节用 `tool_result_text`）。
# # 9. **MCP 工具是异步的**：`agent.invoke()` 会失败，要用 `await agent.ainvoke()`。
# # 10. **多服务端必须处理同名工具**：经典适配器不会自动加前缀（官方新 API 会），
# #     重名会让模型随机挑一个 —— 用 `get_tools(server_name=...)` 分别取名再加前缀。
# # 11. **stdio 传输下服务端是子进程**：脚本路径要用绝对路径；记得进程会在客户端退出时被回收。
# # 12. **压 fastmcp 日志要设它自己的 logger**：`mcp.run(show_banner=False)` +
# #     `logging.getLogger("fastmcp").setLevel(logging.ERROR)`；设 root 级别**无效**。
# # 13. **常驻服务收尾要连子进程树**：`taskkill /F /T /PID`；只杀父进程会留下孤儿吊着端口。
# # 14. **刚被 taskkill 的进程句柄不会立刻释放**：紧跟着删目录会 `WinError 32` ——
# #     重试 + 容忍失败（`remove_temp_dir`），绝不该让演示崩在清理上。
# # 15. **连本机回环服务端前设 `NO_PROXY=127.0.0.1,localhost`**：系统代理会接管回环请求。
# #
# # **第 4 节**
# #
# # 16. `OpenAIEmbeddings` 连第三方兼容端点**必须** `check_embedding_ctx_length=False`，
# #     并给 `request_timeout`。
# # 17. **rerank 分数不能当概率用**，只按排名用。
# # 18. **中文切分必须自定义 `separators`**。
# # 19. **检索结果要带来源 `metadata`**，否则答案无法溯源。
# # 20. **工具化的检索要把返回内容限长**，否则多次调用撑爆上下文。
#

# # ## 官方链接
# #
# # **测试与评估**
# # - 测试总览：<https://docs.langchain.com/oss/python/langchain/test/index>
# # - 单元测试：<https://docs.langchain.com/oss/python/langchain/test/unit-testing>
# # - 集成测试：<https://docs.langchain.com/oss/python/langchain/test/integration-testing>
# # - 评估与轨迹匹配：<https://docs.langchain.com/oss/python/langchain/test/evals>
# #
# # **护栏与运行时**
# # - 护栏（确定性 vs 模型）：<https://docs.langchain.com/oss/python/langchain/guardrails>
# # - Runtime / 依赖注入：<https://docs.langchain.com/oss/python/langchain/runtime>
# # - 工具与保留参数：<https://docs.langchain.com/oss/python/langchain/tools>
# #
# # **MCP**
# # - MCP 总览与传输方式：<https://docs.langchain.com/oss/python/langchain/mcp/index>
# # - 连接生命周期 / 多服务端 / 命名空间：<https://docs.langchain.com/oss/python/langchain/mcp/connections>
# # - 认证：<https://docs.langchain.com/oss/python/langchain/mcp/auth>
# # - DeepAgents 接 MCP 工具：<https://docs.langchain.com/oss/python/deepagents/tools>
# #
# # **RAG**
# # - 知识库（RAG 标准流程）：<https://docs.langchain.com/oss/python/langchain/knowledge-base>
# # - DeepAgents 检索增强（Agentic RAG）：<https://docs.langchain.com/oss/python/deepagents/retrieval>
# # - 文本切分：<https://docs.langchain.com/oss/python/langchain/splitters>
# #
# # **编排**
# # - LangGraph 图 API（LCEL 的官方替代主线）：<https://docs.langchain.com/oss/python/langgraph/graph-api>
# # - 多 Agent 自定义工作流：<https://docs.langchain.com/oss/python/langchain/multi-agent/custom-workflow>
#